In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression



In [2]:
train_path = "../input/osic-pulmonary-fibrosis-progression/train.csv"
test_path = "../input/osic-pulmonary-fibrosis-progression/test.csv"
sub_path = "../input/osic-pulmonary-fibrosis-progression/sample_submission.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
submission = pd.read_csv(sub_path)



In [3]:
submission["Patient"] = submission["Patient_Week"].apply(lambda x: x.split("_")[0])
submission["Weeks"] = submission["Patient_Week"].apply(lambda x: int(x.split("_")[1]))



In [4]:
sub_feat = submission[["Patient", "Weeks"]].merge(
    test, on="Patient", how="left", suffixes=("_sub", "_test")
)

sub_feat["Weeks"] = sub_feat["Weeks_sub"]
sub_feat = sub_feat.drop(columns=["Weeks_sub", "Weeks_test"])



In [5]:
# add quadratic week term
train_feat = train[["Weeks", "Percent", "Age", "Sex", "SmokingStatus"]].copy()
train_feat["Weeks_sq"] = train_feat["Weeks"] ** 2

sub_feat = sub_feat.copy()
sub_feat["Weeks_sq"] = sub_feat["Weeks"] ** 2

combined = pd.concat(
    [
        train_feat,
        sub_feat[["Weeks", "Percent", "Age", "Sex", "SmokingStatus", "Weeks_sq"]],
    ],
    axis=0,
    ignore_index=True,
)

combined_enc = pd.get_dummies(combined, columns=["Sex", "SmokingStatus"])

X_train = combined_enc.iloc[: len(train_feat), :].reset_index(drop=True)
X_sub = combined_enc.iloc[len(train_feat) :, :].reset_index(drop=True)

y_train = train["FVC"]



In [6]:
model = LinearRegression()
model.fit(X_train, y_train)

# compute a data‑driven confidence (MAE, but not lower than 70)
train_pred = model.predict(X_train)
mae = np.mean(np.abs(train_pred - y_train))
conf_val = max(70, mae)



In [7]:
pred_fvc = model.predict(X_sub)



In [8]:
submission["FVC"] = pred_fvc
submission["Confidence"] = conf_val  # calibrated confidence

final_submission = submission[["Patient_Week", "FVC", "Confidence"]]



In [9]:
final_submission.to_csv("submission.csv", index=False)